In [6]:
# comprehensive table with all runs for text cleaning on two rounds: GPT generation and author revision
from pathlib import Path
import pandas as pd

BASE_DIR = Path("/Users/HP/Documents/second-publication/current-analysis/meetingTranscript")
RUN_CSV = BASE_DIR / "semantic-fidelity" / "bertscore_semantic_fidelity.csv"
AUTHOR_CSV = BASE_DIR / "semantic-fidelity" / "bertscore_author_versions.csv"
OUTPUT_TEX = BASE_DIR / "semantic-fidelity" / "text_cleaning_fidelity_table.tex"

# Section selections from the 3 initial runs
RETAINED_RUNS = {
    "elicitation_process": "cleaned_contexts_run02_20260629_104239.json",
    "project_description": "cleaned_contexts_run03_20260629_104252.json",
    "demonstrator_description": "cleaned_contexts_run01_20260629_104226.json",
}

SECTION_ORDER = [
    "elicitation_process",
    "project_description",
    "demonstrator_description",
]

STATUS_ORDER = [
    "Not retained",
    "Retained",
    "Final version",
]

SECTION_LABELS = {
    "elicitation_process": "Elicitation",
    "project_description": "Project",
    "demonstrator_description": "Demonstrator",
}

def simplify_name(name: str) -> str:
    return Path(name).stem

# ---------- Load run-selection rows ----------
run_df = pd.read_csv(RUN_CSV)
run_df = run_df[run_df["section"] != "__overall__"].copy()

run_df["status"] = run_df.apply(
    lambda row: (
        "Retained"
        if RETAINED_RUNS.get(row["section"]) == row["cleaned_file"]
        else "Not retained"
    ),
    axis=1,
)

run_df = run_df.rename(columns={"cleaned_file": "edited_file"})
run_df["raw_file"] = "transcription"

run_df = run_df[
    ["raw_file", "edited_file", "section", "precision", "recall", "f1", "status"]
].copy()

# ---------- Load author-version rows ----------
author_df = pd.read_csv(AUTHOR_CSV)
author_df = author_df[author_df["section"] != "__overall__"].copy()

author_df["status"] = "Final version"

author_df = author_df[
    ["raw_file", "edited_file", "section", "precision", "recall", "f1", "status"]
].copy()

# ---------- Combine ----------
final_df = pd.concat([run_df, author_df], ignore_index=True)

final_df["raw_file"] = final_df["raw_file"].map(simplify_name)
final_df["edited_file"] = final_df["edited_file"].map(simplify_name)

final_df["section"] = pd.Categorical(
    final_df["section"],
    categories=SECTION_ORDER,
    ordered=True,
)

final_df["status"] = pd.Categorical(
    final_df["status"],
    categories=STATUS_ORDER,
    ordered=True,
)

final_df = final_df.sort_values(["section", "status", "edited_file"]).reset_index(drop=True)

for col in ["precision", "recall", "f1"]:
    final_df[col] = final_df[col].round(3)

final_df["section"] = final_df["section"].map(SECTION_LABELS)

# ---------- Export LaTeX ----------
latex_table = final_df.to_latex(
    index=False,
    escape=False,
    caption="BERT-score semantic fidelity assessment for the initial runs and the final revision.",
    label="tab:text_cleaning_semantic_fidelity",
    column_format="ll l c c c l"
)

OUTPUT_TEX.write_text(latex_table, encoding="utf-8")

print(final_df)
print(f"\nLaTeX table written to:\n{OUTPUT_TEX}")

             raw_file                             edited_file       section  \
0       transcription  cleaned_contexts_run01_20260629_104226   Elicitation   
1       transcription  cleaned_contexts_run03_20260629_104252   Elicitation   
2       transcription  cleaned_contexts_run02_20260629_104239   Elicitation   
3   author-review-raw                    author-review-edited   Elicitation   
4       transcription  cleaned_contexts_run01_20260629_104226       Project   
5       transcription  cleaned_contexts_run02_20260629_104239       Project   
6       transcription  cleaned_contexts_run03_20260629_104252       Project   
7   author-review-raw                    author-review-edited       Project   
8       transcription  cleaned_contexts_run02_20260629_104239  Demonstrator   
9       transcription  cleaned_contexts_run03_20260629_104252  Demonstrator   
10      transcription  cleaned_contexts_run01_20260629_104226  Demonstrator   
11  author-review-raw                    author-revi